Passo 1: Bloco de importação de bibliotecas — cole e use-as no topo do seu notebook dataview.ipynb

In [13]:
#Importar as bibliotecas

import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

RF01 – Criar ou Carregar o Dataset de Vendas

O projeto deve usar um dataset de vendas com ao menos as colunas: id | data | cliente | produto | categoria | regiao | quantidade | preco

O dataset deve conter dados intencionalmente sujos: valores None, datas inválidas e espaços extras em strings — para você praticar a limpeza na RF03.

O projeto deverá incluir um dataset de vendas. O estudante pode gerar o dataset sinteticamente no próprio código (recomendado para facilitar a reprodução) ou utilizar um arquivo CSV real. Opção A – Gerar o dataset no código (recomendado):

In [16]:
# Gerar dataset de vendas

def gerar_dataset_vendas(n_registros=1500, seed=42):

    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook","Smartphone","Tablet","Monitor","Teclado","Mouse"]

    precos = {
        "Notebook":3500,
        "Smartphone":2200,
        "Tablet":1800,
        "Monitor":1200,
        "Teclado":250,
        "Mouse":120
    }

    categorias = {
        "Notebook":"Computadores",
        "Smartphone":"Celulares",
        "Tablet":"Celulares",
        "Monitor":"Computadores",
        "Teclado":"Periféricos",
        "Mouse":"Periféricos"
    }

    regioes = ["Sul","Sudeste","Nordeste","Centro-Oeste","Norte"]

    clientes = [f"Cliente_{i:03d}" for i in range(1,31)]

    data_inicio = datetime(2024,1,1)

    dados=[]

    for i in range(n_registros):

        produto = random.choice(produtos)

        dados.append({

            "id":i+1,

            "data":(data_inicio + timedelta(days=random.randint(0,364))).strftime("%Y-%m-%d"),

            "cliente":random.choice(clientes),

            "produto":produto,

            "categoria":categorias[produto],

            "regiao":random.choice(regioes),

            "quantidade":random.randint(1,10),

            "preco":precos[produto]

        })

    df = pd.DataFrame(dados)
    
    return df


In [17]:
# Inserir dados sujos no dataset

df = gerar_dataset_vendas()

# 60 clientes com espaços
idx = np.random.choice(df.index,60,replace=False)
df.loc[idx,"cliente"] = (
" " + df.loc[idx,"cliente"] + " "
)

# 30 regiões com espaços
idx = np.random.choice(df.index,30,replace=False)
df.loc[idx,"regiao"] = (
" " + df.loc[idx,"regiao"]
)

# 5 clientes nulos
idx = np.random.choice(df.index,5,replace=False)
df.loc[idx,"cliente"] = None

# 3 produtos nulos
idx = np.random.choice(df.index,3,replace=False)
df.loc[idx,"produto"] = None

# 10 quantidades nulas
idx = np.random.choice(df.index,10,replace=False)
df.loc[idx,"quantidade"] = None

#Preços nulos
idx = np.random.choice(df.index,5,replace=False)
df.loc[idx,"preco"] = None

#50 datas inválidas
datas_invalidas = [
"31/02/2024",
"99/99/9999",
"abc",
"",
"32/01/2024",
"2024-15-01",
"2024-02-30",
"15-15-2024",
"2024/99/10",
"data_invalida"
]

idx = np.random.choice(df.index,50,replace=False)

for i, linha in enumerate(idx):
    df.loc[linha,"data"] = datas_invalidas[i % len(datas_invalidas)]



# Outliers em preço e quantidade
idx = np.random.choice(df.index, 10, replace=False)
df.loc[idx, "preco"] = np.random.randint(30000, 100000, 10)

idx = np.random.choice(df.index, 20, replace=False)
df.loc[idx, "quantidade"] = np.random.randint(500, 1500, 20)

# Seleciona 50 linhas aleatórias
duplicados = df.sample(n=50, random_state=42)

# Adiciona essas linhas novamente ao DataFrame
df = pd.concat([df, duplicados], ignore_index=True)



print("Dados sujos adicionados com sucesso!")

# Salvar dataset com dados sujos

df.to_csv("../data/raw/vendas_sujo.csv", index=False)

print(df.head())

Dados sujos adicionados com sucesso!
   id        data      cliente     produto     categoria    regiao  \
0   1  2024-02-27  Cliente_001       Mouse   Periféricos  Nordeste   
1   2  2024-03-12  Cliente_024  Smartphone     Celulares       Sul   
2   3  2024-10-29  Cliente_014    Notebook  Computadores       Sul   
3   4  2024-04-21  Cliente_008    Notebook  Computadores     Norte   
4   5  2024-10-14  Cliente_007    Notebook  Computadores     Norte   

   quantidade   preco  
0         4.0   120.0  
1         9.0  2200.0  
2         1.0  3500.0  
3        10.0  3500.0  
4         7.0  3500.0  


RF03 – Limpar e Tratar os Dados O projeto deverá realizar a limpeza dos dados. O estudante deve tratar ao menos os seguintes problemas: Remover ou imputar linhas com valores nulos nas colunas críticas (quantidade, preco_unitario); Remover linhas com datas inválidas; Converter a coluna de data para o tipo datetime; Remover espaços extras em colunas de texto com .str.strip(); Registrar no console quantos registros foram removidos.

In [19]:
# Carregar o dataset

df = pd.read_csv("../data/raw/vendas_sujo.csv")

print("=" * 50)
print("INICIANDO LIMPEZA DOS DADOS")
print("=" * 50)

# Quantidade inicial
registros_iniciais = len(df)


# Remover espaços extras


colunas_texto = [
    "cliente",
    "produto",
    "categoria",
    "regiao"
]

for coluna in colunas_texto:
    df[coluna] = df[coluna].astype(str).str.strip()

# Converter data

df["data"] = pd.to_datetime(df["data"], errors="coerce")


# Remover datas inválidas

antes = len(df)

df = df.dropna(subset=["data"])

datas_removidas = antes - len(df)


# Remover valores nulos


antes = len(df)

df = df.dropna(subset=["cliente","quantidade", "preco", "produto"])

nulos_removidos = antes - len(df)


df = df.drop_duplicates().reset_index(drop=True)
duplicados_removidos = registros_iniciais - len(df) - datas_removidas - nulos_removidos

# ==========================
# Total removido
# ==========================

registros_finais = len(df)

total_removidos = registros_iniciais - registros_finais

# ==========================
# Exibir resumo
# ==========================

print("\nResumo da Limpeza")

print("-" * 30)

print(f"Registros iniciais : {registros_iniciais}")
print(f"Datas inválidas removidas : {datas_removidas}")
print(f"Nulos removidos : {nulos_removidos}")
print(f"Duplicados removidos : {duplicados_removidos}")
print(f"Total removido : {total_removidos}")
print(f"Rsegitros finais : {registros_finais}")

# ==========================
# Salvar dataset limpo
# ==========================

df.to_csv(
    "../data/processed/v1_com_outliers/vendas_limpo.csv",
    index=False
)

print("\nArquivo salvo com sucesso!")
print("../data/processed/v1_com_outliers/vendas_limpo.csv")

INICIANDO LIMPEZA DOS DADOS

Resumo da Limpeza
------------------------------
Registros iniciais : 1550
Datas inválidas removidas : 53
Nulos removidos : 22
Duplicados removidos : 46
Total removido : 121
Rsegitros finais : 1429

Arquivo salvo com sucesso!
../data/processed/v1_com_outliers/vendas_limpo.csv
